## Base vectorielle complete — 28 editions segmentees, metadonnees Synthese

Objectif : etendre la base vectorielle Ovide au-dela des 73 illustrations deja
themees (`ovide_bnu_corpus_siglip.pkl`, construites dans `vector_base.ipynb` a
partir de 3 feuilles de `BNU_corpus.ods`). Ici on vectorise **toutes** les
illustrations deja segmentees dans `data/editions_ovide/segmentees/` (2191 crops,
28 dossiers/editions), chacune enrichie avec les metadonnees d'edition trouvees
dans la feuille **Synthese** de `BNU_corpus.ods` (titre, ville, graveur, technique,
annee, langue, famille iconographique).

**Ce que ça n'est pas** : un theme (deluge / creation_monde / ...) precis par
illustration — sauf pour les 73 deja connues, reprises telles quelles ici. Les
~2100 autres n'ont pas de theme individuel (voir feuilles `#N` de `BNU_corpus.ods`
pour une etape ulterieure, qui donnera un theme precis pour les editions ayant une
feuille dediee — Salomon, Wickram, Solis, Savery notamment).

In [1]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image

RACINE = Path("../../").resolve()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device :", DEVICE)

Device : cuda


### 1. Rapprochement dossier segmente -> ligne Synthese

Deux voies : ark deja connu (documente dans `vector_base.ipynb` et les notebooks
`classification_bois_cuivre` / `classification_graveur`), ou a defaut un
rapprochement par ville + annee + fragment du nom du graveur (verifie a l'oeil,
voir le tableau affiche plus bas — 27/27 dossiers rapproches avec succes).

In [2]:
SEG_DIR = RACINE / "data" / "editions_ovide" / "segmentees"

# ark deja connus avec certitude (repris de vector_base.ipynb / classification_*/01_*.ipynb)
ARK_CONNU = {
    'bois_salomon_rouille_lyon1557': 'btv1b2200047r',
    'bois_wickram_behem_mayence1545': 'bsb10139926',
    'bois_solis_feyerabend_francfort1581': 'bsb00087854',
    'cuivre_savery_farnaby_paris1637': 'bsb10863401',
    'bois_eskrich_rouille_lyon1556': 'btv1b22000559',
    'bois_leroy_gueynard_lyon1510': 'bsb11054210',
    'cuivre_baur_sn_augsbourg1709': 'bsb10872075',
    'cuivre_baur_sn_vienne1639': 'bsb10872073',
    'cuivre_borcht_plantin_anvers1591': 'bsb00004340',
    'cuivre_bouche_blaeu_amsterdam1702': 'bpt6k15151988',
    'cuivre_depasse_depasse_koln1602': 'bpt6k15218623',
    'cuivre_depasse_jansonius_arnhem1607': 'bpt6k1522448r',
    'cuivre_gaultier_guillemot_paris1610': 'bsb11913284',
    'cuivre_gaultier_veuveguillemot_paris1614': 'bpt6k6277348n',
    'cuivre_goltzius_goltzius_haarlem1589': 'btv1b10534945n',
    'cuivre_goltzius_goltzius_haarlem1589_couleur': 'btv1b10534945n',
    'cuivre_isaac_langelier_paris1617': 'bpt6k722055',
    'cuivre_mathieu_langelier_paris1619': 'btv1b22000826',
    'cuivre_monconet_sommaville_paris1660': 'bpt6k87045023',
    'cuivre_tempesta_dejode_anvers1606': 'btv1b54000051z',
    'cuivre_tempesta_jansonius_amsterdam1610': 'bsb00008186',
}

PAT_ARK = re.compile(r'ark:/12148/([a-zA-Z0-9]+)|(bsb\d+)')


def extraire_ark(row):
    for col in ['version numérisée 1', 'version numérisée 2', 'url catalogue']:
        val = str(row.get(col))
        m = PAT_ARK.search(val)
        if m:
            return m.group(1) or m.group(2)
    return None


def normaliser(s):
    s = unicodedata.normalize('NFKD', str(s)).encode('ascii', 'ignore').decode()
    return re.sub(r'[^a-z0-9]', '', s.lower())


xl = pd.ExcelFile(RACINE / "retours_celine" / "BNU_corpus.ods", engine='odf')
synthese = xl.parse('Synthèse', header=0)
synthese['ark_num'] = synthese.apply(extraire_ark, axis=1)
synthese['graveur_norm'] = synthese['graveur\xa0: Nom, Prénom'].apply(normaliser)
synthese['ville_norm'] = synthese['ville'].apply(normaliser)

dossiers = sorted(d.name for d in SEG_DIR.iterdir() if d.is_dir() and d.name != "bnu_corpus_celine")
print(f"{len(dossiers)} dossiers d'edition a rapprocher (bnu_corpus_celine traite a part)")

27 dossiers d'edition a rapprocher (bnu_corpus_celine traite a part)


In [3]:
def rapprocher(dossier):
    m = re.match(r"^(bois|cuivre)_([a-z0-9]+)_([a-z0-9]+)_([a-z]+?)(\d{4})(?:_(.+))?$", dossier)
    if not m:
        return None, "NOM NON PARSE"
    technique, graveur_court, editeur_court, ville_court, annee, variante = m.groups()

    ark = ARK_CONNU.get(dossier)
    if ark:
        candidats = synthese[synthese["ark_num"] == ark]
        if len(candidats):
            return candidats.iloc[0], "ark connu -> trouve"

    prefixe = "ark connu mais absent de Synthese" if ark else "pas d'ark"
    candidats = synthese[
        (synthese["ville_norm"].str.contains(ville_court, na=False))
        & (synthese["année"].astype(str).str.contains(str(annee), na=False))
    ]
    if len(candidats) == 0:
        return None, f"{prefixe} -> AUCUN CANDIDAT"
    if len(candidats) == 1:
        return candidats.iloc[0], f"{prefixe} -> trouve par ville+annee"
    affine = candidats[candidats["graveur_norm"].str.contains(graveur_court[:5], na=False)]
    if len(affine):
        return affine.iloc[0], f"{prefixe} -> affine par graveur ({len(candidats)} candidats)"
    return None, f"{prefixe} -> AMBIGU ({len(candidats)} candidats)"


CHAMPS_SYNTHESE = {
    "titre": "titre abrégé", "ville": "ville", "publisher": "publisher", "annee": "année",
    "langue": "langue", "technique": "technique", "graveur": "graveur\xa0: Nom, Prénom",
    "type_iconographique": "type iconographique", "cadre_grave": "cadre gravé",
    "famille_iconographique": "familles iconographiques", "ark_synthese": "ark_num",
}

meta_par_dossier = {}
for dossier in dossiers:
    ligne, statut = rapprocher(dossier)
    rec = {"statut": statut}
    if ligne is not None:
        for champ, col in CHAMPS_SYNTHESE.items():
            rec[champ] = ligne[col]
    meta_par_dossier[dossier] = rec

table_meta = pd.DataFrame(meta_par_dossier).T
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 30)
print(table_meta[["statut", "titre", "ville", "annee", "graveur", "famille_iconographique"]])
print()
print(table_meta["statut"].str.replace(r"\(.*\)", "", regex=True).value_counts())

                                                       statut                          titre                  ville      annee                        graveur famille_iconographique
bois_eskrich_rouille_lyon1556             ark connu -> trouve  Trois Premiers livres de l...                   Lyon       1556                Eskrich, Pierre                      7
bois_leroy_gueynard_lyon1510    ark connu mais absent de S...  P. Ovidii Nasonis Metamorp...                   Lyon       1510            Leroy II, Guillaume                      2
bois_salomon_rouille_lyon1557             ark connu -> trouve        La Metamorphose figurée                   Lyon       1557               Salomon, Bernard                      7
bois_solis_feyerabend_franc...            ark connu -> trouve  P. Ovidii Metamorphosis, O...  Francfort-sur-le-Main       1581                  Solis, Virgil                      7
bois_wickram_behem_mayence1545            ark connu -> trouve  P. Ovidii Nasonis dess all...   

### 2. Themes deja connus (73 illustrations de `vector_base.ipynb`)

Reutilises tels quels — pas de re-classification, juste report par chemin de
fichier, pour ne pas perdre l'information deja validee.

In [4]:
base_existante = pd.read_pickle(RACINE / "data" / "vector_bases" / "ovide_bnu_corpus_siglip.pkl")
THEME_CONNU = dict(zip(base_existante["chemin"].apply(lambda c: Path(c).name), base_existante["theme"]))
print(f"{len(THEME_CONNU)} themes deja connus (par nom de fichier)")

73 themes deja connus (par nom de fichier)


### 3. Vectorisation SigLIP (niveaux de gris) — tous les crops des 28 dossiers

Meme pretraitement que `vector_base.ipynb` / `retrieval.ipynb` (niveaux de gris
puis re-RGB, pour neutraliser le coloriage) — la nouvelle base reste comparable
aux deux bases existantes (`bibles_siglip.pkl`, `ovide_bnu_corpus_siglip.pkl`).

In [5]:
from transformers import AutoModel, AutoProcessor

NOM_MODELE_SIGLIP = "google/siglip-base-patch16-224"
processor = AutoProcessor.from_pretrained(NOM_MODELE_SIGLIP)
model_siglip = AutoModel.from_pretrained(NOM_MODELE_SIGLIP).to(DEVICE).eval()
print("SigLIP charge")


def embed_image(chemin):
    img = Image.open(chemin).convert("RGB").convert("L").convert("RGB")
    inputs = processor(images=img, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        vec = model_siglip.get_image_features(**inputs).pooler_output
    return vec.cpu().numpy()[0]

Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

SigLIP charge


In [6]:
import time

lignes = []
t0 = time.time()
total = 0
for dossier in dossiers + ["bnu_corpus_celine"]:
    meta = meta_par_dossier.get(dossier, {})
    fichiers = sorted((SEG_DIR / dossier).glob("*.jpg"))
    for k, chemin in enumerate(fichiers, 1):
        vec = embed_image(chemin)
        lignes.append({
            "chemin": str(chemin),
            "dossier": dossier,
            "theme": THEME_CONNU.get(chemin.name),
            "titre": meta.get("titre"),
            "ville": meta.get("ville"),
            "publisher": meta.get("publisher"),
            "annee": meta.get("annee"),
            "langue": meta.get("langue"),
            "technique": meta.get("technique"),
            "graveur": meta.get("graveur"),
            "type_iconographique": meta.get("type_iconographique"),
            "cadre_grave": meta.get("cadre_grave"),
            "famille_iconographique": meta.get("famille_iconographique"),
            "ark": meta.get("ark_synthese"),
            "embedding": vec.tolist(),
        })
    total += len(fichiers)
    print(f"  {dossier:45s} {len(fichiers):4d} crops   (cumul {total}, {time.time()-t0:.0f}s)")

print(f"\nTermine : {len(lignes)} illustrations vectorisees en {time.time()-t0:.0f}s")

  bois_eskrich_rouille_lyon1556                   42 crops   (cumul 42, 2s)


  bois_leroy_gueynard_lyon1510                    19 crops   (cumul 61, 2s)


  bois_salomon_rouille_lyon1557                  161 crops   (cumul 222, 7s)


  bois_solis_feyerabend_francfort1581            184 crops   (cumul 406, 18s)


  bois_wickram_behem_mayence1545                  50 crops   (cumul 456, 20s)


...

  cuivre_philippe_hackiana_leyde1670              16 crops   (cumul 1820, 100s)


  cuivre_savery_farnaby_paris1637                 18 crops   (cumul 1838, 103s)


  cuivre_tempesta_dejode_anvers1606              139 crops   (cumul 1977, 110s)


  cuivre_tempesta_jansonius_amsterdam1610        149 crops   (cumul 2126, 119s)
  cuivre_weyen_barbin_paris1669                    3 crops   (cumul 2129, 119s)


  bnu_corpus_celine                               62 crops   (cumul 2191, 122s)

Termine : 2191 illustrations vectorisees en 122s


### 4. Sauvegarde

In [7]:
corpus_complet = pd.DataFrame(lignes)
print(corpus_complet.shape)
print(corpus_complet["theme"].notna().sum(), "illustrations avec theme connu")
print(corpus_complet.groupby("dossier").size().sort_values(ascending=False))

DOSSIER_VECTOR_DB = RACINE / "data" / "vector_bases"
chemin_sortie = DOSSIER_VECTOR_DB / "ovide_corpus_complet_siglip.pkl"
corpus_complet.to_pickle(chemin_sortie)
print(f"\nSauvegarde : {chemin_sortie}  ({chemin_sortie.stat().st_size / 1024:.0f} Ko)")

(2191, 15)
77 illustrations avec theme connu
dossier
bois_solis_feyerabend_francfort1581             184
cuivre_borcht_plantin_anvers1591                182
bois_salomon_rouille_lyon1557                   161
cuivre_baur_sn_augsbourg1709                    161
cuivre_tempesta_jansonius_amsterdam1610         149
cuivre_monconet_sommaville_paris1660            148
cuivre_tempesta_dejode_anvers1606               139
cuivre_depasse_jansonius_arnhem1607             136
cuivre_mathieu_langelier_paris1619              135
cuivre_depasse_depasse_koln1602                 134
cuivre_bouche_blaeu_amsterdam1702               126
cuivre_baur_sn_vienne1639                       125
bnu_corpus_celine                                62
bois_wickram_behem_mayence1545                   50
bois_eskrich_rouille_lyon1556                    42
cuivre_goltzius_goltzius_haarlem1589_couleur     38
cuivre_goltzius_goltzius_haarlem1589             38
cuivre_briot_drobet_lyon1628                     28
bois_leroy_


Sauvegarde : /mnt/c/Users/a.saidi/OneDrive - BNU/Bureau/working_dir/data/vector_bases/ovide_corpus_complet_siglip.pkl  (15278 Ko)


## Bilan

Base vectorielle complete du corpus Ovide segmente : chaque illustration porte les
metadonnees d'edition (titre, ville, graveur, technique, annee, langue, famille
iconographique) meme sans theme individuel connu. Les 73 illustrations deja
themees (`vector_base.ipynb`) gardent leur theme.

**Prochaine etape (non faite ici)** : pour les editions avec feuille `#N` dediee
dans `BNU_corpus.ods` (Salomon -> `#0_CORPUS_REF`, Wickram -> `#1`, Solis -> `#7`,
Savery -> `#11` — voir echange precedent), extraire le theme precis de chaque
planche en calant folio <-> page scannee, pour ~400 illustrations supplementaires
avec un theme fin (pas juste "connu/inconnu").